In [ ]:
import os
import re
import glob
import pickle
import shutil
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

VOL_SR = 500.0
Z_THR = 3.0
MIN_PEAK_DISTANCE_FRAMES = 1
SOMA_WINDOW_FRAMES = 2
TAIL_RATIO_THR = 0.30
PEAK_NEIGHBOR_RADIUS = 1  # average of two largest frames in [peak-1, peak, peak+1]


def _as_sorted_unique_int(x):
    if x is None:
        return np.array([], dtype=int)
    a = np.asarray(x, dtype=int).ravel()
    if a.size == 0:
        return np.array([], dtype=int)
    return np.unique(a)


def _safe_cal_sr(v, default=30.0):
    try:
        vv = float(v)
        if np.isfinite(vv) and vv > 0:
            return vv
    except Exception:
        pass
    return float(default)


def _suffix_from_pkl_name(path):
    name = os.path.basename(path)
    for pat in (r"final_correct_spike_detection(.*?)\.pkl$", r"spike_detection_refined_new(.*?)\.pkl$"):
        m = re.search(pat, name, flags=re.IGNORECASE)
        if m:
            s = m.group(1)
            return "main" if s == "" else s
    return os.path.splitext(name)[0]


def _natural_pkl_key(path):
    name = os.path.basename(path)
    m = re.search(r"(final_correct_spike_detection|spike_detection_refined_new)(.*?)\.pkl$", name, flags=re.IGNORECASE)
    if not m:
        return (2, 1, name.lower())
    prefix, suffix = m.group(1).lower(), m.group(2)
    pref_rank = 0 if prefix.startswith("final_correct") else 1
    if suffix == "":
        return (pref_rank, 0, -1)
    try:
        return (pref_rank, 0, int(suffix))
    except Exception:
        return (pref_rank, 1, suffix.lower())


def _find_spike_pkls(cell_folder):
    final_pkls = glob.glob(os.path.join(cell_folder, "final_correct_spike_detection*.pkl"))
    if final_pkls:
        return sorted(list(dict.fromkeys(final_pkls)), key=_natural_pkl_key)
    refined_pkls = glob.glob(os.path.join(cell_folder, "spike_detection_refined_new*.pkl"))
    if refined_pkls:
        return sorted(list(dict.fromkeys(refined_pkls)), key=_natural_pkl_key)
    return []


def _calc_z_trace(x):
    x = np.asarray(x, float)
    mu = float(np.nanmean(x))
    sd = float(np.nanstd(x))
    if (not np.isfinite(sd)) or sd <= 0:
        med = float(np.nanmedian(x))
        mad = float(np.nanmedian(np.abs(x - med)))
        sd = 1.4826 * mad if mad > 0 else np.nan
        mu = med
    if (not np.isfinite(sd)) or sd <= 0:
        return np.zeros_like(x, dtype=float), mu, sd
    return (x - mu) / sd, mu, sd


def _event_bounds_from_z(z, peak_idx):
    n = z.size
    s = int(peak_idx)
    e = int(peak_idx)

    while s > 0 and z[s] > 0:
        s -= 1
    while e < n - 1 and z[e] > 0:
        e += 1

    if e < s:
        e = s
    return s, e


def _peak_amplitude_two_largest(cal, peak_idx, radius=1):
    n = cal.size
    p = int(peak_idx)
    lo = max(0, p - int(radius))
    hi = min(n, p + int(radius) + 1)
    seg = np.asarray(cal[lo:hi], float)
    seg = seg[np.isfinite(seg)]
    if seg.size == 0:
        return 0.0
    k = min(2, seg.size)
    top = np.sort(seg)[-k:]
    amp = float(np.mean(top))
    return max(0.0, amp)


def _exp_decay(t, a, b, c):
    return a * np.exp(-b * t) + c


def _apply_tail_overlap_rules(events, cal, cal_sr, ratio_thr=0.3):
    if len(events) == 0:
        return events

    # initialize corrected amplitude = raw amplitude
    for ev in events:
        ev["amp_corrected"] = float(ev["amp_raw"])
        ev["tail_ratio"] = np.nan
        ev["tail_pred_at_peak"] = np.nan
        ev["exclude_reason"] = ""
        ev["include"] = True

    for i in range(1, len(events)):
        prev = events[i - 1]
        cur = events[i]

        p_prev = int(prev["peak_idx"])
        s_cur = int(cur["start_idx"])
        p_cur = int(cur["peak_idx"])

        if s_cur <= p_prev:
            # overlapping/invalid ordering: keep and skip correction
            cur["include"] = True
            cur["exclude_reason"] = "overlap_no_tail"
            continue

        tail = np.asarray(cal[p_prev:s_cur + 1], float)
        tail = tail[np.isfinite(tail)]
        cur_max = float(cur["peak_raw"]) if np.isfinite(cur["peak_raw"]) else np.nan

        if tail.size == 0 or (not np.isfinite(cur_max)) or cur_max <= 0:
            cur["include"] = True
            cur["exclude_reason"] = "tail_or_peak_invalid"
            continue

        tail_min = float(np.nanmin(tail))
        ratio = float(tail_min / cur_max)
        cur["tail_ratio"] = ratio

        if ratio >= float(ratio_thr):
            cur["include"] = False
            cur["amp_corrected"] = 0.0
            cur["exclude_reason"] = "tail_ratio_ge_thr"
            continue

        # ratio < threshold: fit exp decay on previous event tail and subtract at current peak
        fit_start = int(prev["peak_idx"])
        fit_end = int(prev["end_idx"])
        fit_end = max(fit_start + 2, fit_end)
        fit_end = min(fit_end, cal.size - 1)

        y = np.asarray(cal[fit_start:fit_end + 1], float)
        t = (np.arange(y.size, dtype=float) / float(cal_sr))
        good = np.isfinite(y)
        y = y[good]
        t = t[good]

        pred = np.nan
        if y.size >= 3:
            a0 = max(1e-6, float(y[0] - y[-1]))
            b0 = 1.0
            c0 = float(y[-1])
            try:
                popt, _ = curve_fit(
                    _exp_decay,
                    t,
                    y,
                    p0=[a0, b0, c0],
                    bounds=([0.0, 0.0, -np.inf], [np.inf, np.inf, np.inf]),
                    maxfev=20000,
                )
                dt = float((p_cur - fit_start) / float(cal_sr))
                pred = float(_exp_decay(np.array([dt], dtype=float), *popt)[0])
            except Exception:
                pred = np.nan

        cur["tail_pred_at_peak"] = pred
        if np.isfinite(pred):
            cur["amp_corrected"] = max(0.0, float(cur["amp_raw"]) - float(pred))
            cur["exclude_reason"] = "exp_subtracted"
        else:
            cur["amp_corrected"] = max(0.0, float(cur["amp_raw"]))
            cur["exclude_reason"] = "exp_fit_failed"

    return events


def detect_calcium_events(trace_cal, cal_sr, z_thr=3.0, min_peak_distance=1,
                          peak_neighbor_radius=1, ratio_thr=0.3,
                          soma_peak_frames=None, soma_window_frames=2):
    cal = np.asarray(trace_cal, float).ravel()
    n = cal.size
    if n == 0:
        return []

    z, z_mu, z_sd = _calc_z_trace(cal)

    peaks, props = find_peaks(
        z,
        height=float(z_thr),
        distance=max(1, int(min_peak_distance)),
    )

    events = []
    for p in peaks.tolist():
        s, e = _event_bounds_from_z(z, p)
        amp_raw = _peak_amplitude_two_largest(cal, p, radius=int(peak_neighbor_radius))

        ev = {
            "start_idx": int(s),
            "peak_idx": int(p),
            "end_idx": int(e),
            "peak_raw": float(cal[p]) if np.isfinite(cal[p]) else np.nan,
            "peak_z": float(z[p]) if np.isfinite(z[p]) else np.nan,
            "amp_raw": float(max(0.0, amp_raw)),
            "z_mu": float(z_mu) if np.isfinite(z_mu) else np.nan,
            "z_sd": float(z_sd) if np.isfinite(z_sd) else np.nan,
            "include": True,
            "exclude_reason": "",
            "local_dSpike": np.nan,
        }
        events.append(ev)

    events = sorted(events, key=lambda x: x["start_idx"])
    events = _apply_tail_overlap_rules(events, cal, cal_sr, ratio_thr=ratio_thr)

    # Optional local-dSpike classification if soma events are provided
    if soma_peak_frames is not None:
        soma = _as_sorted_unique_int(soma_peak_frames)
        w = int(max(0, soma_window_frames))
        for ev in events:
            p = int(ev["peak_idx"])
            has_soma = np.any((soma >= p - w) & (soma <= p + w))
            ev["local_dSpike"] = bool(not has_soma)

    return events


def _events_to_df(events, cell_folder, pkl_name, cal_sr):
    rows = []
    for i, ev in enumerate(events):
        rows.append({
            "cell_folder": cell_folder,
            "pkl_name": pkl_name,
            "event_idx": i,
            "start_idx": int(ev["start_idx"]),
            "peak_idx": int(ev["peak_idx"]),
            "end_idx": int(ev["end_idx"]),
            "start_t_s": float(ev["start_idx"]) / float(cal_sr),
            "peak_t_s": float(ev["peak_idx"]) / float(cal_sr),
            "end_t_s": float(ev["end_idx"]) / float(cal_sr),
            "peak_z": float(ev["peak_z"]),
            "peak_raw": float(ev["peak_raw"]),
            "amp_raw": float(ev["amp_raw"]),
            "amp_corrected": float(ev.get("amp_corrected", ev["amp_raw"])),
            "tail_ratio": float(ev["tail_ratio"]) if np.isfinite(ev.get("tail_ratio", np.nan)) else np.nan,
            "tail_pred_at_peak": float(ev["tail_pred_at_peak"]) if np.isfinite(ev.get("tail_pred_at_peak", np.nan)) else np.nan,
            "include": bool(ev.get("include", True)),
            "exclude_reason": str(ev.get("exclude_reason", "")),
            "local_dSpike": ev.get("local_dSpike", np.nan),
        })
    return pd.DataFrame(rows)


def plot_calcium_event_overlay(cell_folder, pkl_path, trace_vol, trace_cal, events,
                               cal_sr, vol_sr=500.0):
    trace_vol = np.asarray(trace_vol, float).ravel()
    trace_cal = np.asarray(trace_cal, float).ravel()

    vol_t = np.arange(trace_vol.size, dtype=float) / float(vol_sr)
    cal_t = np.arange(trace_cal.size, dtype=float) / float(cal_sr)

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Scatter(x=vol_t, y=trace_vol, mode="lines", name="Voltage trace",
                   line=dict(color="#f4a261", width=1.0)),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(x=cal_t, y=trace_cal, mode="lines", name="Calcium trace",
                   line=dict(color="gray", width=1.0)),
        secondary_y=True,
    )

    if len(events) > 0:
        s_idx = np.array([int(ev["start_idx"]) for ev in events], dtype=int)
        e_idx = np.array([int(ev["end_idx"]) for ev in events], dtype=int)
        p_idx = np.array([int(ev["peak_idx"]) for ev in events], dtype=int)
        inc = np.array([bool(ev.get("include", True)) for ev in events], dtype=bool)

        # event boundaries (all detected)
        fig.add_trace(
            go.Scatter(x=cal_t[s_idx], y=trace_cal[s_idx], mode="markers", name="Event start",
                       marker=dict(color="black", size=9, symbol="x")),
            secondary_y=True,
        )
        fig.add_trace(
            go.Scatter(x=cal_t[e_idx], y=trace_cal[e_idx], mode="markers", name="Event end",
                       marker=dict(color="brown", size=9, symbol="x")),
            secondary_y=True,
        )

        # included vs excluded peaks
        if np.any(inc):
            pi = p_idx[inc]
            fig.add_trace(
                go.Scatter(x=cal_t[pi], y=trace_cal[pi], mode="markers", name="Event peak (included)",
                           marker=dict(color="purple", size=10, symbol="x")),
                secondary_y=True,
            )
        if np.any(~inc):
            pe = p_idx[~inc]
            fig.add_trace(
                go.Scatter(x=cal_t[pe], y=trace_cal[pe], mode="markers", name="Event peak (excluded)",
                           marker=dict(color="red", size=9, symbol="x")),
                secondary_y=True,
            )

    suffix = _suffix_from_pkl_name(pkl_path)
    out_base = os.path.join(cell_folder, f"calcium_event_overlay_{suffix}")

    fig.update_layout(
        template="simple_white",
        title=(
            f"{os.path.basename(cell_folder)} | {os.path.basename(pkl_path)}"
            f"<br><sup>Calcium events via find_peaks z>={Z_THR}, overlap ratio rule={TAIL_RATIO_THR}</sup>"
        ),
        width=1400,
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig.update_xaxes(title_text="Time (s)")
    fig.update_yaxes(title_text="Voltage", secondary_y=False)
    fig.update_yaxes(title_text="Calcium (dF/F)", secondary_y=True)

    html_path = out_base + ".html"
    svg_path = out_base + ".svg"
    fig.write_html(html_path)
    svg_saved = True
    try:
        fig.write_image(svg_path)
    except Exception as e:
        svg_saved = False
        print(f"[WARN] SVG export failed ({os.path.basename(svg_path)}): {e}")

    return {
        "figure": fig,
        "html_path": html_path,
        "svg_path": svg_path,
        "svg_saved": svg_saved,
        "suffix": suffix,
    }


def _ensure_main_overlay_copy(cell_folder, out_infos):
    target_html = os.path.join(cell_folder, "calcium_event_overlay_main.html")
    target_svg = os.path.join(cell_folder, "calcium_event_overlay_main.svg")

    if not out_infos:
        return None

    chosen = None
    for oi in out_infos:
        if oi.get("suffix") == "main":
            chosen = oi
            break
    if chosen is None:
        chosen = out_infos[0]

    try:
        src_html = os.path.abspath(chosen["html_path"])
        dst_html = os.path.abspath(target_html)
        if os.path.exists(src_html) and src_html != dst_html:
            shutil.copy2(src_html, dst_html)

        src_svg = os.path.abspath(chosen["svg_path"])
        dst_svg = os.path.abspath(target_svg)
        if os.path.exists(src_svg) and src_svg != dst_svg:
            shutil.copy2(src_svg, dst_svg)
    except Exception as e:
        print(f"[WARN] failed to copy main calcium overlay: {e}")

    return target_html


In [ ]:
# -------------------------------
# Run calcium-event detection for all PyrLowFR cells
# -------------------------------
DB_PATH = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv"
MAX_CELLS = None  # set int for quick test

DB = pd.read_csv(DB_PATH)
if MAX_CELLS is not None:
    DB = DB.iloc[:int(MAX_CELLS)].copy()

all_events = []

for row_idx, row in DB.iterrows():
    cell_folder = str(row["Link"])
    cal_sr = _safe_cal_sr(row.get("CALsr", 30.0), default=30.0)

    if not os.path.isdir(cell_folder):
        print(f"[SKIP] missing folder: {cell_folder}")
        continue

    pkl_list = _find_spike_pkls(cell_folder)
    if len(pkl_list) == 0:
        print(f"[SKIP] no spike pkl found: {cell_folder}")
        continue

    print("\n" + "=" * 110)
    print(f"Cell {row_idx + 1}/{len(DB)} | {cell_folder} | CALsr={cal_sr} | pkls={len(pkl_list)}")

    out_infos = []

    for pkl_path in pkl_list:
        try:
            with open(pkl_path, "rb") as f:
                d = pickle.load(f)

            trace_cal = np.asarray(d.get("trace_cal", []), dtype=float).ravel()
            trace_vol = np.asarray(d.get("trace_vol", []), dtype=float).ravel()
            if trace_cal.size == 0 or trace_vol.size == 0:
                print(f"[SKIP] empty trace in {os.path.basename(pkl_path)}")
                continue

            # Optional soma frames if available in future; currently None in this DB
            soma_frames = d.get("soma_cal_peak_frames", None)

            events = detect_calcium_events(
                trace_cal,
                cal_sr=cal_sr,
                z_thr=Z_THR,
                min_peak_distance=MIN_PEAK_DISTANCE_FRAMES,
                peak_neighbor_radius=PEAK_NEIGHBOR_RADIUS,
                ratio_thr=TAIL_RATIO_THR,
                soma_peak_frames=soma_frames,
                soma_window_frames=SOMA_WINDOW_FRAMES,
            )

            edf = _events_to_df(events, cell_folder, os.path.basename(pkl_path), cal_sr)
            if len(edf):
                edf["suffix"] = _suffix_from_pkl_name(pkl_path)
                all_events.append(edf)

            out = plot_calcium_event_overlay(
                cell_folder=cell_folder,
                pkl_path=pkl_path,
                trace_vol=trace_vol,
                trace_cal=trace_cal,
                events=events,
                cal_sr=cal_sr,
                vol_sr=VOL_SR,
            )
            out_infos.append(out)

            suffix = _suffix_from_pkl_name(pkl_path)
            csv_out = os.path.join(cell_folder, f"calcium_events_by_{suffix}.csv")
            edf.to_csv(csv_out, index=False)

            n_inc = int(edf["include"].sum()) if len(edf) else 0
            n_exc = int((~edf["include"]).sum()) if len(edf) else 0
            print(
                f"[OK] {os.path.basename(pkl_path)} | detected={len(edf)} | included={n_inc} | excluded={n_exc} | "
                f"saved={os.path.basename(out['html_path'])}"
            )

        except Exception as e:
            print(f"[ERROR] {pkl_path}: {e}")

    main_overlay = _ensure_main_overlay_copy(cell_folder, out_infos)
    if main_overlay:
        print(f"[MAIN-OVERLAY] saved: {main_overlay}")

if len(all_events) == 0:
    raise RuntimeError("No calcium events detected across PyrLowFR cells.")

events_df = pd.concat(all_events, ignore_index=True)
summary_csv = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_calcium_events_detected.csv"
events_df.to_csv(summary_csv, index=False)

print("\nDone.")
print(f"Total detected events: {len(events_df)}")
print(f"Included events: {int(events_df['include'].sum())}")
print(f"Excluded events: {int((~events_df['include']).sum())}")
print(f"Saved combined table: {summary_csv}")